In [ ]:
import json
from dotenv import load_dotenv

import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

from weather_tool import get_weekend_weather
from email_sender_tool import send_email

In [ ]:
from IPython.display import display, Markdown, HTML

In [ ]:
import asyncio
from datetime import datetime
from zoneinfo import ZoneInfo

from agents import Agent, Runner, WebSearchTool, function_tool

In [ ]:
load_dotenv(override=True)

In [ ]:
today = datetime.now(ZoneInfo("Europe/Madrid")).strftime("%A %d de %B de %Y")

In [ ]:
RULES = """
- Busca planes con niños en Alcobendas, San Sebastián de los Reyes, Tres Cantos y el norte de Madrid Ciudad.
- Busca planes como mercados o ferias que puedan disfrutar también los adultos.
- Los planes no tienen porque ser válidos para todos los niños de la familia, pero sí para los niños. Indica la edad mínima recomendada.
- Si el clima acompaña, puedes ofrecer planes de parques, excursiones o visitas a lugares exteriores.
- Si la previsión del tiempo indica que no habrá precipitaciones, ofrece antes una lista de planes como parques o excursiones al aire libre a menos de 1 hora de Alcobendas.
- Ofrece todas las opciones que encuentres por dia, no es necesario un plan al dia.
- Focalizate en los planes que son especificos de ese viernes, sábado y domingo como teatros, conciertos, talleres, actividades de ocio y cultura, etc. No inventes planes, solo los que existan.
- Adapta los planes al tiempo: exterior si hay poca probabilidad de lluvia, interior si no. Distingue entre mañana y tarde.
- Identifica los planes que son gratis y los que son de pago. Indica si hay que reservar.
- SIEMPRE que exista una web a la que acudir a buscar más información sobre el plan, inclúyela. No inventes webs.
"""

INSTRUCTIONS = f"""
Hoy es {today}. Ayudas a familias a encontrar planes que hacer este viernes por la tarde, sábado y domingo.

## Reglas
{RULES}

## Flujo de trabajo
1. Primero consulta la previsión del fin de semana con get_weekend_weather.
2. Después busca planes en internet y adáptalos al tiempo.
3. Devuelve un resumen de la previsión meteorológica y una lista de planes para el fin de semana, divididos por día y bloque (mañana/tarde).
4. Utiliza send_email para mandar el resumen de la previsión meteorológica y la lista de planes a los destinatarios configurados en las variables de entorno.
El asunto del email debe ser 'Planes con niños para este fin de semana' y el cuerpo del email debe contener la una introducción sobre los planes
(no indiques para cuantos niños), la previsión meteorológica y la lista de planes en formato HTML.

SIEMPRE indica que este resumen ha sido generado por un asistente virtual y que la información puede no ser precisa. Recomienda a los destinatarios que verifiquen la información antes de acudir a los planes.
SIEMPRE ofrece todos los planes que encuentres en tu busqueda.

NUNCA respondas con un plan de interior inventado, puedes inventar planes de parque. Si no hay planes disponibles, indica que no hay planes disponibles para el fin de semana.
NUNCA confirmes que se ha mandado un mail que no se ha mandado. Si no hay planes disponibles, no envíes un mail.
"""


In [ ]:
from email_sender_tool import send_email

agent = Agent(
    name="Planes con niños",
    instructions=INSTRUCTIONS,
    model="gpt-6-luna",
    tools=[get_weekend_weather, WebSearchTool(), send_email],
)

In [ ]:
for t in [get_weekend_weather, WebSearchTool(), send_email]:
    print(type(t))

In [ ]:
result = await Runner.run(agent, "¿Qué podemos hacer este fin de semana con dos niños de 2 y 5 años?")

In [ ]:
display(Markdown(result.final_output))